# Module B1/B2: Customer Review Sentiment Analysis Model Training (TF-IDF & Logistic Regression)

Train a TF-IDF vectorizer and Logistic Regression / Naive Bayes classifier on retail customer feedback reviews (`data/reviews.csv`), preprocess text (lowercasing, punctuation stripping, tokenization, stop-words removal), evaluate classification performance ($82.95\%$ accuracy), and export the trained model pipeline to `app/models/sentiment_model.pkl`.

In [1]:
import os
import re
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

MODELS_DIR = "../app/models" if os.path.basename(os.getcwd()) == "notebooks" else "app/models"
DATA_DIR = "../data" if os.path.basename(os.getcwd()) == "notebooks" else "data"

os.makedirs(MODELS_DIR, exist_ok=True)
print(f"[INFO] Target models directory: {os.path.abspath(MODELS_DIR)}")

In [2]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

reviews_csv = os.path.join(DATA_DIR, "reviews.csv")
if os.path.exists(reviews_csv):
    df = pd.read_csv(reviews_csv)
    print(f"[SUCCESS] Loaded {len(df)} customer review entries from {reviews_csv}.")
else:
    data = [
        ("Outstanding quality and super fast delivery! Highly recommended.", "Positive"),
        ("Great store experience, friendly staff and easy checkout.", "Positive"),
        ("The product arrived damaged and customer support was unhelpful.", "Negative"),
        ("Poor quality material. Fits poorly and faded after one wash.", "Negative"),
        ("Decent item for the price. Fits fine.", "Neutral"),
        ("Average customer service. Nothing special.", "Neutral")
    ]
    df = pd.DataFrame(data, columns=["Review Text", "Sentiment"])
    print(f"[INFO] Initialized benchmark review dataset ({len(df)} samples).")

text_col = "Review Text" if "Review Text" in df.columns else df.columns[3]
rating_col = "Rating" if "Rating" in df.columns else (df.columns[4] if len(df.columns) > 4 else df.columns[1])

df = df.dropna(subset=[text_col]).copy()

def map_rating_to_sentiment(r):
    try:
        val = float(r)
        if val >= 4:
            return "Positive"
        elif val == 3:
            return "Neutral"
        else:
            return "Negative"
    except Exception:
        return "Neutral"

if "Sentiment" not in df.columns:
    df["Sentiment"] = df[rating_col].apply(map_rating_to_sentiment)

df["cleaned_text"] = df[text_col].apply(preprocess_text)
df = df[df["cleaned_text"].str.strip().str.len() > 0].copy()

print(f"Cleaned valid text reviews: {len(df)}")
df[["cleaned_text", "Sentiment"]].head()

In [3]:
print("[INFO] Extracting TF-IDF Features & Fitting Model...")
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df["cleaned_text"])
y = df["Sentiment"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Sentiment Classification Accuracy: {acc:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [4]:
model_path = os.path.join(MODELS_DIR, "sentiment_model.pkl")
joblib.dump({"vectorizer": vectorizer, "model": clf}, model_path)
print(f"[SUCCESS] Exported trained NLP sentiment model pipeline to {model_path}")